In [36]:
import langchain
langchain.__version__

'1.3.2'

In [37]:
import os
from dotenv import load_dotenv

load_dotenv()

api_key=os.getenv("GOOGLE_API_KEY")

print(api_key[:5] + "..." if api_key else "Not set")
langchain_key = os.getenv("LANGCHAIN_API_KEY")
print("Using key:", langchain_key[:12] + "..." if langchain_key else "Not set")

In [26]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model='gemini-3.5-flash')
response=llm.invoke("how are you")

print(response)

content=[{'type': 'text', 'text': "I'm doing well, thank you for asking! How are you doing today? How can I help you?", 'extras': {'signature': 'Et0HCtoHAQw51se+3DOsiPqNXmVhzMsv1v9K+T4bSbpob+vr3U4md/VOCWJ6gHiIV4wV0P7CfgkfEZ/0TpqG86gZnic5I7UCdcqcOyWm64ovtrRxtd2/ACJh1ZVsOCFVT1nDJeynlCl+4+PrvPT7BodUoN414FlU1C3yP8R10n1byzuC30dACINahzKVNmHvga7ntoKAA2flXjfnT+YR6XSzsRfsN9oPIc33+2QWkloRQFXj6lsoHIls5yWsBhHYq38IW3mA6PyOPfKc0bc6hEGWLplruRVG59dbenBXhC9sjLefD7ptdUIGK9aXw9oZTXAtpWFdD1AszomnERh/wDVLpMLQCOLEwH/wh2aeE77sIdH5tW+CoDbzbhkPSvhXg/HEKN+6tG6N6tZYiRKTP+2lDrOLFelL6zMXes7+tRlSfJjXVHS1cdnGT5yHVdLLncqCfFJKLQ5GUbzTPdTCvdqXrdbZaAGnH7+H7LBFl+3+JzaKQq7g4dzAvg5U0LxliBK7JL0rabjr+CcdE6WICOJRZVpar6goAgevCgvaoawVaoFlbncCXfNBn5ylxdbTInMpbjOK5NfcC+jeukQkizlxrTMp95TxhBkRVgR02gq0GAqxlsn3AMg8lhoChux9F0X/wWBEgVOaGWdlE6R34i7SUs/dARfwClsGr2E2OAFEGXPpMkQFsoCb11W+yXr9jM9FHvk0/ky5MXGVZixULY06GuVi/rsxoSmuGhB4SiKwjZXdEJXTQkGccnGGxoEOXcFmQQiPtRSkJiocaXuyO+7Ml7gjmeD1HW5Vpx4EknqvzAW8TmmtdZrbDuzO77YoS14undG6+99Yy83830sPJaC

In [27]:
from langchain_core.output_parsers import StrOutputParser

output_parsers=StrOutputParser()
output_parsers.invoke(response)


"I'm doing well, thank you for asking! How are you doing today? How can I help you?"

In [28]:
# simle chain

chain=llm | output_parsers
chain.invoke("Tell me a joke")

'Why did the scarecrow win an award?\n\nBecause he was outstanding in his field!'

In [29]:
from typing import List
from pydantic import BaseModel, Field

class MobileReview(BaseModel):
    phone_model: str = Field(description="Name and model of the phone")
    rating: float = Field(description="Overall rating out of 5")
    pros: List[str] = Field(description="List of positive aspects")
    cons: List[str] = Field(description="List of negative aspects")
    summary: str = Field(description="Brief summary of the review")

review_text = """
Just got my hands on the new Galaxy S21 and wow, this thing is slick! The screen is gorgeous,
colors pop like crazy. Camera’s insane too, especially at night - my Insta game’s never been
stronger. Battery life’s solid, lasts me all day no problem.

Not gonna lie though, it’s pretty pricey. And what’s with ditching the charger? C’mon Samsung.
Also, still getting used to the new button layout, keep hitting Bixby by mistake.

Overall, I’d say it’s a solid 4 out of 5. Great phone, but a few annoying quirks keep it from
being perfect. If you’re due for an upgrade, definitely worth checking out!
"""

structured_llm = llm.with_structured_output(MobileReview)
output = structured_llm.invoke(review_text)
output

MobileReview(phone_model='Galaxy S21', rating=4.0, pros=['Gorgeous, vibrant screen', 'Insane camera, especially for night photography', 'Solid, all-day battery life'], cons=['High price tag', 'No charger included in the box', 'Button layout leads to accidental Bixby triggers'], summary='A great phone with an excellent screen and camera, though held back from perfection by its price, lack of an included charger, and minor button quirks.')

In [30]:
output.pros

['Gorgeous, vibrant screen',
 'Insane camera, especially for night photography',
 'Solid, all-day battery life']

In [31]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("Tell me a joke about {topic}")
prompt.invoke({"topic" : "Programmer"})

ChatPromptValue(messages=[HumanMessage(content='Tell me a joke about Programmer', additional_kwargs={}, response_metadata={})])

In [32]:
chain = prompt | llm | output_parsers
chain.invoke({"topic" : "Life"})

'Adult life is basically just saying, **"Things are pretty crazy right now, but they should settle down in a couple of weeks,"** over and over again until you die.'

In [33]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import HumanMessage, SystemMessage

System_msg=SystemMessage(content="You are a assistant that tells dark joke")
human_msg=HumanMessage(content="Tell me a joke about women")

llm.invoke([System_msg, human_msg])

AIMessage(content=[{'type': 'text', 'text': 'I asked my wife if she’d ever consider getting a divorce.\n\nShe smiled and said, "No, darling. Murder, yes. Divorce, never."', 'extras': {'signature': 'EvQZCvEZAQw51serGGBiY514tlpo+bSMmR2bSwlI3ISJMnZ+3M6uTFT2+kZaRGO+13/tJK/9wNyuSTng7kNhtRspzIchTP6u5oA0qS2E8WK/oM6UXc5NC5nB3PIsgOTL/wwvIOge+dEuPaascEXt9p4sxsUH5YZc6vuu8tETAMS2BPYHU0btpGyy1yLxNtcGJw3W8vap7BCAAXLcSAeWk1UxTw80I30BELPvMASoa74RKcEwxmrbk5G9NBnPH/N5lWyO5Sc0GsJO1utvgIAoSjMa+GjJIE6XM5iid9r9YPhX6EHtvItkToHzgWPCjRQvRM1w2SKPAipzdrR/XyXSmlR/erpcubMsSAORBpNRwCXpTFaXXLm1oftqPKh3YVl+SvjW5Q4KvItJpYNfJ84lBhinhs6LmKVu0JLojlqzNmBRQSZtPGUptnc0btYKd6rCm8nRetqxmiYGUohfNQ0FxhOaVTd3tAkTVZ6VJCjIKg2+iUoBsZ2f57+csvFqFFBFMkUWfKuScpyohtUNB6V4Iim0MPshifMKY9Jba1PdLjpgNAtbzZc2XiVCfD0jqFaZ3VuHAjsg4cLBJ+noOXvlPTK6mq3V52w3mP4K5Yn/UPjqR0mBxMXOe9O5CKqmDfarxjUTNCSNysMmQe1gcvET4jT9coRUcDRXmzPMk54luFfV3kY6srd8ISRL6K0S8cthbCnKZyb89kLgHrjwmYlc853w80hOm7RBRdZGXeUCeeD+rNLlhM96kwHWRfCdW4krWmj5tTQifInugIMe3rIxilldXAosbqEJj5

In [44]:
# proper chain 

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser 

prompt = ChatPromptTemplate.from_template("Tell me a short joke about {topic}")

llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash")

output_parsers= StrOutputParser()

chain = prompt | llm | output_parsers

result = chain.invoke({"topic":"Genz"})
print(result)

How many Gen Zers does it take to change a lightbulb?

None. They’ll just sit in the dark and film a TikTok about how the vibe is "lowkey depressing, fr fr."


In [47]:
## LLM Messages

from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import HumanMessage, SystemMessage

System_msg = SystemMessage(content = "You are a helpful assistant who give shprt jokes")
Human_Message = HumanMessage(content = "Tell me something about programming")
llm.invoke([System_msg, Human_Message])

AIMessage(content=[{'type': 'text', 'text': 'Why do programmers prefer dark mode?\n\nBecause light attracts bugs!', 'extras': {'signature': 'Ep0LCpoLAQw51sdIF6agKc2u+kDy9IbQ3ItGC2G6VHay6d2WrT6v5eSYwIBT4EQuYY4YvqSUFkwFX/epYZhIDVhbIe0/OIJpj9xpVKwYiqz19nhTUflxMnTefQSMdlUT7KTwd9I9G1QrewCVWXqkU4I8wd38HJqi1rWt8bEGrRex4hYM1m1aDwlbT8grdeD4xVRjU7sRprbJssevLfMmi1UH+qwpGWHKUdsHtyxCo55F3re++Hcu1Z2/0sy/m1UGxxYfJQnABktiFgrepuGLyWcnXAMZjbzr84eCSlUd85nVQLy9+xHMxaTRp/tgEUogH5lxF/IfJna7C9SF7ENfregaUACRFWR7peOZY9/155ef5T6Hl1lbinf5N7RIGbrX/cTYHWD3XFj0ArQk8+PfUo4ZDiGqrAip9xyMRx5PjmyuD10x7UzpIsiwiSogY+zf47DGxWWVh7o39KTr3HPI6YOiQzpqCKGWsqnzhfQhsHEN+hmKC5WOu64Ka8jmBDNqEUHcfdqod0MX95z6movyfTUC4lfubpw5HvdWNgr6/us9AIIpgn4rRMm2sO5PBRnCDAmEn7PUCYYuwdVYd8bRCHLZdTqRMKmfkY4nOPmGApEU5bApbl5X+G20ncIVEnzYriKS8GlGfLbl/yGZbFb21O870A7spXlFJXSL73JE6YU41G4zZdxswm5XN8dGTXV9B9luODWEyIlUbKOts5ZlX/46QUm2J+DVyRnKAYZ6QEGl8ZHdOAKpiwzzefLJpcs9CD18e63G8+iVeORRzVLEZ3TfjuzDVSboR6oHmrUudhRm36zHjIUais4eccLosoKsF/7wee/s8npXtE5EVZrHqhGPFzK

In [2]:
from langchain_community.document_loaders import PyPDFLoader, Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
import os
from typing import List
from langchain_core.documents import Document


def load_doc(folder_path:str) -> List[Document]:
    Documents=[]
    for files in os.listdir(folder_path):
        file_path = os.path.join(folder_path, files)

        if files.endswith('.pdf'):
            loader = PyPDFLoader(file_path)
        elif files.endswith('.docx'):
            loader = Docx2txtLoader(file_path)
        else:
            print(f"Unsupported file type: {files}")
            continue
        Documents.extend(loader.load())
    return Documents
folder_path="D:/project/experiment/content"
Documents = load_doc(folder_path)
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200, length_function=len)
print(f"Loaded {len(Documents)} documents from the folder.")

splits = text_splitter.split_documents(Documents)
print(f"Split the documents into {len(splits)} chunks.")


Loaded 4 documents from the folder.
Split the documents into 5 chunks.


In [3]:
from langchain_community.embeddings.sentence_transformer import SentenceTransformerEmbeddings
embedding_function = SentenceTransformerEmbeddings(model_name="all-MiniLM-L6-v2")
document_embeddings = embedding_function.embed_documents([split.page_content for split in splits])
document_embeddings[0]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2732.77it/s]


[0.0293357465416193,
 -0.027797531336545944,
 -0.01846340484917164,
 -0.058924250304698944,
 0.08951887488365173,
 -0.026082225143909454,
 -0.10806205123662949,
 0.03759301081299782,
 -0.005008654668927193,
 -0.041687216609716415,
 -0.016387708485126495,
 -0.02612929232418537,
 -0.0313599668443203,
 0.010636315681040287,
 -0.07282692939043045,
 -0.0030148024670779705,
 -0.01876574382185936,
 -0.01873071677982807,
 0.0665455013513565,
 -0.07033520936965942,
 0.003246106207370758,
 -0.01593754068017006,
 0.06552284955978394,
 0.014457416720688343,
 -0.011876809410750866,
 0.10337897390127182,
 -0.005314045585691929,
 0.001742130727507174,
 0.0036549982614815235,
 -0.03610217198729515,
 -0.012576743960380554,
 0.04590792953968048,
 0.030862504616379738,
 -0.05400508642196655,
 0.02634744718670845,
 0.08197516947984695,
 -0.03140775114297867,
 -0.04810931161046028,
 0.04922591149806976,
 -0.0024446044117212296,
 0.005876422859728336,
 -0.01402465533465147,
 -0.003966060001403093,
 -0.05092

In [4]:
from langchain_chroma import Chroma

collection_name = "my_collection"
vectorstore = Chroma.from_documents(collection_name=collection_name, documents=splits, embedding=embedding_function, persist_directory="./chroma_db")
#db.persist()

print("Vector store created and persisted to './chroma_db'")

Vector store created and persisted to './chroma_db'


In [5]:
#LEts check

query = "When was GreenGrow Innovations founded?"
search_results = vectorstore.similarity_search(query, k=2)

print(f"\nTop 2 most relevant chunks for the query: '{query}'\n")
for i, result in enumerate(search_results, 1):
    print(f"Result {i}:")
    print(f"Source: {result.metadata.get('source', 'Unknown')}")
    print(f"Content: {result.page_content}")
    print()


Top 2 most relevant chunks for the query: 'When was GreenGrow Innovations founded?'

Result 1:
Source: D:/project/experiment/content\GreenGrow Innovations_ Company History.docx
Content: GreenGrow Innovations was founded in 2010 by Sarah Chen and Michael Rodriguez, two agricultural engineers with a passion for sustainable farming. The company started in a small garage in Portland, Oregon, with a simple mission: to make farming more environmentally friendly and efficient.



In its early days, GreenGrow focused on developing smart irrigation systems that could significantly reduce water usage in agriculture. Their first product, the WaterWise Sensor, was launched in 2012 and quickly gained popularity among local farmers. This success allowed the company to expand its research and development efforts.



By 2015, GreenGrow had outgrown its garage origins and moved into a proper office and research facility in the outskirts of Portland. This move coincided with the development of their se